# Problem 1

In [9]:
import math

from assignment_3_q_1 import (
    loadConll2003Dataset,
    preprocessDatasetForGlove,
    loadGloveWikiGigaword100,
    SOS_TOKEN,
    EOS_TOKEN,
    UNK_TOKEN,
)
# return - a list of sentence dictionaries
def parseConllFile(filePath, labelToId):
  examples = []
  with open(filePath, encoding="utf-8") as textStream:
    tokens = []
    labels = []
    for rawLine in textStream:
      line = rawLine.strip()
      if not line or line.startswith("-DOCSTART-"):
        if tokens:
          examples.append({"tokens": tokens, "ner_tags": labels})
          tokens = []
          labels = []
        continue

      fields = line.split()
      tokens.append(fields[0])
      labels.append(labelToId[fields[-1]])

    if tokens:
      examples.append({"tokens": tokens, "ner_tags": labels})
  return examples

  
# Load the local CoNLL-2003 train, validation, and test splits
# datasetDirectory - directory containing train.txt, valid.txt, and test.txt
# return - dataset dictionary with splits and BIO label names
def loadConll2003Dataset(datasetDirectory):
  labelNames = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
  labelToId = {labelName: labelIndex for labelIndex, labelName in enumerate(labelNames)}
  return {
    "train": parseConllFile(f"{datasetDirectory}/train.txt", labelToId),
    "validation": parseConllFile(f"{datasetDirectory}/valid.txt", labelToId),
    "test": parseConllFile(f"{datasetDirectory}/test.txt", labelToId),
    "labelNames": labelNames,
  }


# Build a short summary of the dataset splits
# dataset - dictionary returned by loadConll2003Dataset
# return - a printable summary string
def formatDatasetSummary(dataset):
  return "\n".join([
    f"train sentences: {len(dataset['train'])}",
    f"validation sentences: {len(dataset['validation'])}",
    f"test sentences: {len(dataset['test'])}",
    f"BIO labels: {', '.join(dataset['labelNames'])}",
  ])

# a.)

In [10]:
# CoNLL token -> GloVe key if present, else UNK.
def mapTokenForGlove(token, embedding):
  token = token.lower()
  return token if token in embedding else UNK_TOKEN


# For every split: GloVe OOV -> UNK, wrap tokens with <S> / </S>
def preprocessDatasetForGlove(dataset, embedding):
  
  labelToId = {name: i for i, name in enumerate(dataset["labelNames"])} #BIO label to class Id 
  oId = labelToId["O"]  #outside the name span 

  processed = {"labelNames": dataset["labelNames"]} #will hold the processed dataset

  for split in ("train", "validation", "test"):
    processed_sentences = []
    for sent in dataset[split]:
      mapped = [mapTokenForGlove(token, embedding) for token in sent["tokens"]]
      processed_sentences.append({
        "tokens": [SOS_TOKEN] + mapped + [EOS_TOKEN],
        "ner_tags": [oId] + list(sent["ner_tags"]) + [oId],
      })
    processed[split] = processed_sentences
  return processed

# b. )

In [11]:
def stripBound(example):
    """Drop SOS/EOS tokens and matching boundary tags if present."""
    tokens, tags = example["tokens"], example["ner_tags"]
    if tokens and tokens[0] == SOS_TOKEN and tokens[-1] == EOS_TOKEN:
        return tokens[1:-1], tags[1:-1]
    return tokens, tags


def trainHmm(train_examples, label_names, smoothing=1.0):
    alpha = smoothing
    n_tags = len(label_names)

    stripped = [stripBound(ex) for ex in train_examples]
    stripped = [(t, y) for t, y in stripped if t]

    vocab = {UNK_TOKEN}
    for tokens, _ in stripped:
        vocab.update(tokens)
    vocab_list = sorted(vocab)
    vocab_size = len(vocab_list)
    word_to_ix = {w: i for i, w in enumerate(vocab_list)}

    pi_counts = [alpha] * n_tags
    trans_counts = [[alpha] * n_tags for _ in range(n_tags)]
    emit_counts = [[alpha] * vocab_size for _ in range(n_tags)]

    for tokens, tags in stripped:
        pi_counts[tags[0]] += 1
        for y, w in zip(tags, tokens):
            emit_counts[y][word_to_ix[w]] += 1
        for y_prev, y_next in zip(tags, tags[1:]):
            trans_counts[y_prev][y_next] += 1

    pi_den = sum(pi_counts)
    pi = [c / pi_den for c in pi_counts]

    A = []
    for row in trans_counts:
        den = sum(row)
        A.append([c / den for c in row])

    B = []
    for row in emit_counts:
        den = sum(row)
        B.append([c / den for c in row])

    return {
        "label_names": label_names,
        "n_tags": n_tags,
        "pi": pi,
        "A": A,
        "B": B,
        "word_to_ix": word_to_ix,
        "vocab_size": vocab_size,
    }

In [ ]:
def viterbiDecode(words, model):
    """Best tag sequence; converts pi/A/B to logs inside (avoids underflow)."""
    if not words:
        return []

    n_tags = model["n_tags"]
    pi = model["pi"]
    A = model["A"]
    B = model["B"]
    word_to_ix = model["word_to_ix"]
    unk_ix = word_to_ix[UNK_TOKEN]

    log_pi = [math.log(p) for p in pi]
    log_A = [[math.log(a) for a in row] for row in A]
    log_B = [[math.log(b) for b in row] for row in B]

    obs_ix = [word_to_ix[w] if w in word_to_ix else unk_ix for w in words]
    T = len(obs_ix)

    dp = [[-math.inf] * n_tags for _ in range(T)]
    back = [[0] * n_tags for _ in range(T)]

    for s in range(n_tags):
        dp[0][s] = log_pi[s] + log_B[s][obs_ix[0]]

    for t in range(1, T):
        for s in range(n_tags):
            best_val = -math.inf
            best_prev = 0
            emit = log_B[s][obs_ix[t]]
            for sp in range(n_tags):
                cand = dp[t - 1][sp] + log_A[sp][s] + emit
                if cand > best_val:
                    best_val = cand
                    best_prev = sp
            dp[t][s] = best_val
            back[t][s] = best_prev

    best_last = max(range(n_tags), key=lambda s: dp[T - 1][s])
    path = [0] * T
    path[T - 1] = best_last
    for t in range(T - 2, -1, -1):
        path[t] = back[t + 1][path[t + 1]]
    return path


def evaluateHmmTagger(model, examples):
    total_correct = 0
    total_tokens = 0
    for ex in examples:
        tokens, gold = stripBound(ex)
        if not tokens:
            continue
        pred = viterbiDecode(tokens, model)
        total_correct += sum(int(p == g) for p, g in zip(pred, gold))
        total_tokens += len(gold)
    return total_correct / total_tokens if total_tokens else 0.0

In [ ]:
# Train / decode (GloVe preprocessing + HMM)
dataset_dir = "data/conll2003"
dataset = loadConll2003Dataset(dataset_dir)
embedding = loadGloveWikiGigaword100()
preprocessed = preprocessDatasetForGlove(dataset, embedding)

hmm = trainHmm(preprocessed["train"], preprocessed["label_names"])
print("validation token accuracy:", evaluateHmmTagger(hmm, preprocessed["validation"]))
print("test token accuracy:", evaluateHmmTagger(hmm, preprocessed["test"]))

# Problem 2

In [ ]:

#################
### Load Data ###
#################

# Parse one CoNLL-U file into sentence records
# datasetDirectory - directory containing the local UD files
# fileName - one split file name
# lowercaseWords - whether to lowercase tokens during parsing
# return - list of parsed sentence dictionaries
def parseConlluFile(datasetDirectory, fileName, lowercaseWords=True):
  filePath = f"{datasetDirectory}/{fileName}"
  sentences = []
  comments = {}
  words = ["<ROOT>"]
  posTags = ["<ROOT_POS>"]
  heads = [-1]
  relations = ["<ROOT_REL>"]

  with open(filePath, encoding="utf-8") as textStream:
    for rawLine in textStream:
      line = rawLine.rstrip("\n")
      if not line:
        if len(words) > 1:
          sentences.append(buildSentenceRecord(words, posTags, heads, relations, comments))
        comments = {}
        words = ["<ROOT>"]
        posTags = ["<ROOT_POS>"]
        heads = [-1]
        relations = ["<ROOT_REL>"]
        continue

      if line.startswith("#"):
        if " = " in line:
          commentKey, commentValue = line[1:].split(" = ", 1)
          comments[commentKey.strip()] = commentValue.strip()
        continue

      fields = line.split("\t")
      tokenId = fields[0]
      if "-" in tokenId or "." in tokenId:
        continue

      words.append(fields[1].lower() if lowercaseWords else fields[1])
      posTags.append(fields[3])
      heads.append(int(fields[6]))
      relations.append(fields[7])

  if len(words) > 1:
    sentences.append(buildSentenceRecord(words, posTags, heads, relations, comments))
  return sentences


# Build one normalized sentence record
# words - token list including the root token
# posTags - POS tags including the root tag
# heads - gold head indices including the root placeholder
# relations - gold dependency relations including the root relation
# comments - metadata collected from CoNLL-U comments
# return - one sentence dictionary
def buildSentenceRecord(words, posTags, heads, relations, comments):
  return {
    "words": list(words),
    "posTags": list(posTags),
    "heads": list(heads),
    "relations": list(relations),
    "length": len(words) - 1,
    "text": comments.get("text", " ".join(words[1:])),
    "sentId": comments.get("sent_id", ""),
  }


# Load the local UD train, dev, and test splits
# datasetDirectory - directory containing the local UD files
# trainFile - training split filename
# devFile - development split filename
# testFile - test split filename
# lowercaseWords - whether to lowercase tokens during parsing
# return - dataset dictionary with three splits
def loadUdDataset(datasetDirectory, trainFile, devFile, testFile, lowercaseWords=True):
  return {
    "training": parseConlluFile(datasetDirectory, trainFile, lowercaseWords),
    "dev": parseConlluFile(datasetDirectory, devFile, lowercaseWords),
    "test": parseConlluFile(datasetDirectory, testFile, lowercaseWords),
  }


#########################
### Projective Filter ###
#########################

# Check whether one gold dependency tree is projective
# sentence - one parsed UD sentence
# return - True if the tree is projective, else False
def isProjective(sentence):
  arcs = []
  for dependentId in range(1, sentence["length"] + 1):
    headId = sentence["heads"][dependentId]
    left = min(headId, dependentId)
    right = max(headId, dependentId)
    arcs.append((left, right, headId, dependentId))

  for arcIndex, firstArc in enumerate(arcs):
    firstLeft, firstRight, firstHead, firstDependent = firstArc
    for secondArc in arcs[arcIndex + 1:]:
      secondLeft, secondRight, secondHead, secondDependent = secondArc
      if len({firstHead, firstDependent, secondHead, secondDependent}) < 4:
        continue
      if firstLeft < secondLeft < firstRight < secondRight:
        return False
      if secondLeft < firstLeft < secondRight < firstRight:
        return False
  return True


# Keep only the projective sentences in one split
# split - list of parsed sentences
# return - kept sentences and counts
def filterProjectiveSentences(split):
  projectiveSentences = [sentence for sentence in split if isProjective(sentence)]
  return {
    "sentences": projectiveSentences,
    "keptCount": len(projectiveSentences),
    "droppedCount": len(split) - len(projectiveSentences),
  }


####################
### Parser State ###
####################

# Build the initial parser state for one sentence
# sentence - one parsed UD sentence
# return - initial arc-standard parser state
def buildInitialParserState(sentence):
  return {
    "stack": [0],
    "buffer": list(range(1, sentence["length"] + 1)),
    "predictedHeads": [-1] + [None] * sentence["length"],
    "arcs": [],
  }


# Check whether parsing is complete
# state - current parser state
# return - True if the parse is complete, else False
def isTerminalState(state):
  return not state["buffer"] and state["stack"] == [0]


# List the valid transitions for the current state
# state - current parser state
# return - list of valid transition names
def getValidTransitions(state):
  transitions = []
  if state["buffer"]:
    transitions.append("SHIFT")
  if len(state["stack"]) >= 2:
    if state["stack"][-2] != 0:
      transitions.append("LEFT_ARC")
    transitions.append("RIGHT_ARC")
  return transitions


# Copy a parser state
# state - current parser state
# return - independent copy of the state
def copyParserState(state):
  return {
    "stack": list(state["stack"]),
    "buffer": list(state["buffer"]),
    "predictedHeads": list(state["predictedHeads"]),
    "arcs": list(state["arcs"]),
  }


# Apply one transition to a parser state
# state - current parser state
# transition - transition to apply
# return - next parser state
def applyTransition(state, transition):
  nextState = copyParserState(state)

  if transition == "SHIFT":
    nextState["stack"].append(nextState["buffer"].pop(0))
    return nextState

  stackTop = nextState["stack"][-1]
  stackSecond = nextState["stack"][-2]

  if transition == "LEFT_ARC":
    nextState["predictedHeads"][stackSecond] = stackTop
    nextState["arcs"].append((stackTop, stackSecond))
    del nextState["stack"][-2]
    return nextState

  if transition == "RIGHT_ARC":
    nextState["predictedHeads"][stackTop] = stackSecond
    nextState["arcs"].append((stackSecond, stackTop))
    nextState["stack"].pop()
    return nextState

  raise ValueError(f"Unknown transition: {transition}")


########################
### Gold Transitions ###
########################

# Build the gold child list for each head token
# sentence - one parsed UD sentence
# return - list of child-id lists
def buildGoldChildren(sentence):
  childLists = [[] for _ in range(sentence["length"] + 1)]
  for dependentId in range(1, sentence["length"] + 1):
    childLists[sentence["heads"][dependentId]].append(dependentId)
  return childLists


# Check whether all gold children of a token have been attached
# tokenId - token whose children we are checking
# state - current parser state
# goldChildren - gold child lists for the sentence
# return - True if all gold children are attached, else False
def allChildrenAttached(tokenId, state, goldChildren):
  return all(state["predictedHeads"][childId] is not None for childId in goldChildren[tokenId])


# Choose the gold next transition under a static oracle
# state - current parser state
# sentence - one parsed UD sentence
# goldChildren - gold child lists for the sentence
# return - gold transition label or None
def getGoldTransition(state, sentence, goldChildren):
  if len(state["stack"]) >= 2:
    stackTop = state["stack"][-1]
    stackSecond = state["stack"][-2]

    if stackSecond != 0 and sentence["heads"][stackSecond] == stackTop:
      if allChildrenAttached(stackSecond, state, goldChildren):
        return "LEFT_ARC"

    if sentence["heads"][stackTop] == stackSecond:
      if allChildrenAttached(stackTop, state, goldChildren):
        return "RIGHT_ARC"

  if state["buffer"]:
    return "SHIFT"
  return None


# Simulate the gold transition sequence for one sentence
# sentence - one parsed UD sentence
# return - parser states and gold transitions, or None if the oracle fails
def simulateGoldTransitions(sentence):
  state = buildInitialParserState(sentence)
  goldChildren = buildGoldChildren(sentence)
  stateExamples = []
  transitions = []

  while not isTerminalState(state):
    transition = getGoldTransition(state, sentence, goldChildren)
    if transition is None:
      return None
    stateExamples.append(copyParserState(state))
    transitions.append(transition)
    state = applyTransition(state, transition)

  return {
    "states": stateExamples,
    "transitions": transitions,
  }

# tokenId - token index or None
# return - token word or a null marker
def getWord(sentence, tokenId):
  if tokenId is None:
    return "<NULL>"
  return sentence["words"][tokenId]


# Get the POS tag for one token id, with a null fallback
# sentence - one parsed UD sentence
# tokenId - token index or None
# return - token POS tag or a null marker
def getPosTag(sentence, tokenId):
  if tokenId is None:
    return "<NULL_POS>"
  return sentence["posTags"][tokenId]


# Read a token id from the top of the stack
# state - current parser state
# relativeIndex - 0 for the top item, 1 for the next item, and so on
# return - token id or None
def getStackToken(state, relativeIndex):
  if len(state["stack"]) > relativeIndex:
    return state["stack"][-1 - relativeIndex]
  return None


# Read a token id from the buffer
# state - current parser state
# bufferIndex - index into the buffer
# return - token id or None
def getBufferToken(state, bufferIndex):
  if len(state["buffer"]) > bufferIndex:
    return state["buffer"][bufferIndex]
  return None


In [ ]:

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

##########################
### Feature Extraction ###
##########################

# Get the word for one token id, with a null fallback
# sentence - one parsed UD sentence


# Build feat for one parser state
# state - current parser state
# sentence - one parsed UD sentence
# return - sparse feature dictionary
def extractParserFeat(state, sentence):

  feat = {}

  stack0 = getStackToken(state, 0) # top of the stack
  stack1 = getStackToken(state, 1) # second top of the stack
  buffer0 = getBufferToken(state, 0) # first item in the buffer
  buffer1 = getBufferToken(state, 1) # second item in the buffer

  feat["stack0_word"] = getWord(sentence, stack0)
  feat["stack0_pos"] = getPosTag(sentence, stack0)
  feat["stack1_word"] = getWord(sentence, stack1)
  feat["stack1_pos"] = getPosTag(sentence, stack1)

  feat["buffer0_word"] = getWord(sentence, buffer0)
  feat["buffer0_pos"] = getPosTag(sentence, buffer0)
  feat["buffer1_word"] = getWord(sentence, buffer1)
  feat["buffer1_pos"] = getPosTag(sentence, buffer1)

  feat["stack_size"] = len(state["stack"])
  feat["buffer_size"] = len(state["buffer"])

  feat["stack0_buffer0_pos"] = (
    getPosTag(sentence, stack0) + "_" + getPosTag(sentence, buffer0)
  )
  feat["stack1_stack0_pos"] = (
    getPosTag(sentence, stack1) + "_" + getPosTag(sentence, stack0)
  )

  return feat


# Collect supervised transition examples from one split
# split - list of parsed projective sentences
# return - feature dictionaries and labels
def collectTransitionExamples(split):
  # Each call to simulateGoldTransitions converts one gold dependency tree
  # into parser states paired with gold next-transition labels.
  featureRows = []
  labels = []

  for sentence in split:
    oracleResult = simulateGoldTransitions(sentence)
    if oracleResult is None:
      continue
    for state, transition in zip(oracleResult["states"], oracleResult["transitions"]):
      featureRows.append(extractParserFeat(state, sentence))
      labels.append(transition)

  return {
    "feat": featureRows,
    "labels": labels,
  }


########################
### Train Classifier ###
########################

# Fit the transition classifier
# trainingFeat - parser-state feature dictionaries
# trainingLabels - gold transition labels
# cValue - inverse regularization strength for logistic regression
# config - configuration dictionary
# return - fitted vectorizer and classifier
def fitTransitionClassifier(trainingFeat, trainLabs, cValue, config):
  vectorizer = DictVectorizer(sparse=True)
  trainMat = vectorizer.fit_transform(trainingFeat)
  classifier = LogisticRegression(
    C=cValue,
    max_iter=config["classifierMaxIter"],
    tol=1e-3,
    random_state=config["randomState"],
  )
  classifier.fit(trainMat, trainLabs)
  return {
    "vectorizer": vectorizer,
    "classifier": classifier,
  }


# Choose the best C value on the dev set
# trainingSplit - projective training sentences
# devSplit - projective dev sentences
# config - configuration dictionary
# return - best model bundle and dev-set results
def selectTransitionClassifier(trainingSplit, devSplit, config):
  #1 . 
  trainEx = collectTransitionExamples(trainingSplit) 
  devEx = collectTransitionExamples(devSplit)

  results = []
  bestRes = None
    #2. 
  for C in config["cCandidates"]:
    modBund = fitTransitionClassifier(    #train a logistic regression model for each C value
      trainEx["feat"],
      trainEx["labels"],
      C,
      config,
    )
    #3. 
    devMat = modBund["vectorizer"].transform(devEx["feat"])
    preds = modBund["classifier"].predict(devMat)
    #4. 
    corr = 0
    for predLab, goldLab in zip(preds, devEx["labels"]):
      if predLab == goldLab:
        corr += 1

    acc = corr / len(devEx["labels"]) if devEx["labels"] else 0.0

    result = {
      "cValue": C,
      "accuracy": acc,
      "modelBundle": modBund,
    }
    results.append(result)
    #5. BEST MODEL 
    if bestRes is None or acc > bestRes["accuracy"]:
      bestRes = result

  return {
    "results": results,
    "bestResult": bestRes,
  }


#######################
### Greedy Decoding ###
#######################

# Predict the best valid transition for the current state
# state - current parser state
# sentence - one parsed UD sentence
# modBund - fitted vectorizer and classifier
# return - predicted transition label
def predictBestTransition(state, sent, modBund):
  # TODO: complete this function for Question 2 Part 3.
  # Recommended steps:
  # 1. extract feat for the current parser state
  # 2. transform those feat with the trained DictVectorizer
  # 3. score all transitions with the classifier
  # 4. call getValidTransitions(state) to find the legal transitions
  # 5. return the highest-scoring transition that is valid in the current state
  #1. 
  feat = extractParserFeat(state, sent)
  #2. 
  featMat = modBund["vectorizer"].transform([feat])
  #3. 
  clf = modBund["classifier"]
  scores = clf.predict_proba(featMat)[0]
  validTrans = set(getValidTransitions(state))

  bestTrans = None
  bestScr = -1.0

  for label, score in zip(clf.classes_, scores):
    if label in validTrans and score > bestScr:
      bestTrans = label
      bestScr = score

  if bestTrans is None:
    bestTrans = getValidTransitions(state)[0]

  return bestTrans


# Greedily decode one sentence
# sentence - one parsed UD sentence
# modBund - fitted vectorizer and classifier
# return - predicted heads and transition sequence
def greedyDecodeSentence(sent, modBund):
  state = buildInitialParserState(sent)
  transLst = []

  while not isTerminalState(state):
    trans = predictBestTransition(state, sent, modBund)
    transLst.append(trans)
    state = applyTransition(state, trans)

  return {
    "predictedHeads": state["predictedHeads"],
    "transitions": transLst,
  }


# Greedily decode one full split
# split - list of parsed sentences
# modBund - fitted vectorizer and classifier
# return - list of predicted parses
def parseSentenceSplit(split, modBund):
  return [greedyDecodeSentence(s, modBund) for s in split]


##################
### Evaluation ###
##################

# Evaluate predicted parses against the gold trees
# sentences - gold parsed sentences
# predictedParses - predicted parse records
# return - UAS, exact match, and token count
def evaluateParses(sents, predParses):
  nTok = 0
  nHead = 0
  nExact = 0

  for sent, predPars in zip(sents, predParses):
    sentOk = True
    for tokId in range(1, sent["length"] + 1):
      nTok += 1
      if predPars["predictedHeads"][tokId] == sent["heads"][tokId]:
        nHead += 1
      else:
        sentOk = False
    if sentOk:
      nExact += 1

  return {
    "uas": nHead / nTok if nTok else 0.0,
    "exactMatch": nExact / len(sents) if sents else 0.0,
    "totalTokens": nTok,
  }


# Format one projective-filter summary line
# splitName - name of the split
# summary - output of filterProjectiveSentences
# return - printable summary line
def formatProjectiveSummary(splitName, summary):
  return (
    f"{splitName}: kept {summary['keptCount']} projective sentences, "
    f"dropped {summary['droppedCount']} non-projective sentences"
  )


# Format one parse-evaluation line
# splitName - name of the split
# metrics - parse evaluation metrics
# return - printable summary line
def formatParseMetrics(splitName, metrics):
  return (
    f"{splitName}: "
    f"UAS={metrics['uas']:.4f}, "
    f"exactMatch={metrics['exactMatch']:.4f}, "
    f"tokens={metrics['totalTokens']}"
  )





In [4]:
############
### Main ###
############


dataDir = "data/ud_english_ewt"
config = {
"cCandidates": [0.25, 1.0, 4.0],
"classifierMaxIter": 500,
"randomState": 42,
}
dataset = loadUdDataset(
dataDir,
"en_ewt-ud-train.conllu",
"en_ewt-ud-dev.conllu",
"en_ewt-ud-test.conllu",
True,
)

trainSumm = filterProjectiveSentences(dataset["training"])
devSumm = filterProjectiveSentences(dataset["dev"])
testSumm = filterProjectiveSentences(dataset["test"])

print("Projective filtering summary:")
print(formatProjectiveSummary("train", trainSumm))
print(formatProjectiveSummary("dev", devSumm))
print(formatProjectiveSummary("test", testSumm))
print()

selRes = selectTransitionClassifier(trainSumm["sentences"], devSumm["sentences"], config)
bestMod = selRes["bestResult"]["modelBundle"]
testPars = parseSentenceSplit(testSumm["sentences"], bestMod)
testMet = evaluateParses(testSumm["sentences"], testPars)

print(f"Selected C: {selRes['bestResult']['cValue']}")
print(formatParseMetrics("test", testMet))


Projective filtering summary:
train: kept 12266 projective sentences, dropped 278 non-projective sentences
dev: kept 1970 projective sentences, dropped 31 non-projective sentences
test: kept 2051 projective sentences, dropped 26 non-projective sentences



/home/bdaqiq/research/Natural-Language-Processing/HW3/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/bdaqiq/research/Natural-Language-Processing/HW3/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data 

Selected C: 4.0
test: UAS=0.7299, exactMatch=0.4047, tokens=24433
